In [ ]:
# fix imports
import os
import sys

module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)

In [ ]:
from src import utils

In [ ]:
from huggingface_hub import HfFolder, login

api_file = "/home/fre.gilad/source/llm-iml/HF_KEY.txt"
hf_token = utils.api_key_from_file(api_file)

HfFolder.save_token(hf_token)
login(token=hf_token)

In [ ]:
import pandas as pd
from src.data import DF_Batcher

data = pd.read_csv("/home/fre.gilad/source/llm-iml/data/HarmBench/harmful_behaviors.csv")
data = data.rename(columns={"goal": "prompt"})

ds_train = data.copy()
ds_eval = data.copy()

dl_train = DF_Batcher(ds_train, batch_size=10, shuffle=True)
dl_eval = DF_Batcher(ds_eval, batch_size=200, shuffle=False)

In [ ]:
print("Train dataset size:", len(ds_train))
print("Eval dataset size:", len(ds_eval))

In [ ]:
from src.eval.hb_evaluator import HarmbenchEvaluator
from src.eval.llama_evaluator import LlamaEvaluator
from src.eval.template_evaluator import TemplateEvaluator
from src.eval.llama_guard_evaluator import LlamaGuardEvaluator
from src.inference.configs import ServeConfig, LLMConfig


evaluators = [
    HarmbenchEvaluator(
        serve_config=ServeConfig(gpu_ids=[1], startup_timeout=5 * 60, client_timeout=60),
        use_context=False,
        silent=False,
    ),
    # LlamaGuardEvaluator(
    #     serve_config=ServeConfig(gpu_ids=[1], startup_timeout=5 * 60, client_timeout=60),
    #     model_name="meta-llama/Meta-Llama-Guard-2-8B",
    #     silent=False,
    # ),
    # LlamaEvaluator(
    #     serve_config=ServeConfig(gpu_ids=[1], startup_timeout=5 * 60, client_timeout=60),
    #     silent=False,
    # ),
    TemplateEvaluator(
        silent=False,
    ),
]

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

torch.set_float32_matmul_precision("high")  # negligable effect

# model_name = "Qwen/Qwen3-0.6B"
# model_name = "GraySwanAI/Llama-3-8B-Instruct-RR"
# model_name = "meta-llama/Llama-3.2-1B-Instruct"
model_name = "meta-llama/Meta-Llama-3-8B-Instruct"
# model_name = "meta-llama/Llama-2-7b-chat-hf"
# model_name = "lmsys/vicuna-7b-v1.5" # TODO: not instruct model, no chat template
# model_name = "mistralai/Mistral-7B-Instruct-v0.3"
# model_name = "tiiuae/falcon-7b-instruct"
# model_name = "mosaicml/mpt-7b-chat"
# model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

utils.set_seed(42)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="cuda:0",
    # attn_implementation="flash_attention_2"
    # attn_implementation="sdpa",
)

if not tokenizer.pad_token:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
print(model)

In [ ]:
from torch import optim
from src.activation_extractor import ActivationExtractor
from src.attacks.optim_attack import OptimAttack
from src.iml_attack import IML_Attack, StopCriteria
from src.adver_model import AdverModel


adv_model = AdverModel(
    model=model,
    tokenizer=tokenizer,
    num_tokens=20,
)

internal_attack = OptimAttack(
    adv_model,
    optim_factory=lambda params: optim.AdamW(params, lr=1e-3),
    steps=30,
    mixed_precision=False,
)

iml_attack = IML_Attack(
    adv_model=adv_model,
    internal_attack=internal_attack,
    optim_factory=lambda params: optim.AdamW(params, lr=2e-2),
    evaluators=evaluators,
    pred_kwargs={"max_length": 200},
    mixed_precision=False,
)

stop = StopCriteria(
    max_epochs=5,
    max_time=15 * 60,
    patience=3,
)

In [ ]:
adv_model = iml_attack.fit(dl_train, dl_eval, stop_criteria=stop)

In [ ]:
adv_model.set_embeddings(iml_attack.best_embeds)
preds = iml_attack.predict(adv_model, dl_eval, max_length=300)
dl_eval.set_column("response", preds)

for i in range(len(preds)):
    print(f" == Prompt:")
    print(ds_eval.iloc[i]["prompt"])
    print(f" == Target:")
    print(ds_eval.iloc[i]["target"])
    print(f" == Prediction:")
    print(preds[i])
    print("\n" + "=" * 50 + "\n")